In [1]:
import os
import json
import glob
import hashlib
from copy import deepcopy
from pathlib import Path
from scipy.io import loadmat
import numpy as np

In [2]:
nickname = "yellow"   
T_in = 9          
mat_path = "/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/cmats/real_data_DEEPSUM_12subj_9reps_scale3_20251020.mat"
base_config_path = "/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/DeepSUM_config_DWIredo.json"
root_runs_dir = "/misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/"
band_name = "NIR"

num_subj = 12
train_subj = 10
registration_ref = 1  
shift_abs_threshold = 4 

scale = 3

manual_numband = None
manual_hiresl = None    
manual_slices = None

recompute_patch_stats = True

In [3]:
def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

def canonical_json_hash(data):
    payload = json.dumps(data, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(payload.encode("utf-8")).hexdigest()

def mean_std_from_npy(files):
    total_n = 0
    total_sum = 0.0
    total_sq_sum = 0.0

    for fp in files:
        arr = np.asarray(np.load(fp, allow_pickle=True), dtype=np.float64)
        total_n += arr.size
        total_sum += float(arr.sum())
        total_sq_sum += float(np.square(arr).sum())

    if total_n == 0:
        raise ValueError("No pixels found in files.")

    mean = total_sum / total_n
    var = (total_sq_sum / total_n) - (mean ** 2)
    var = max(var, 0.0)
    std = float(np.sqrt(var))
    return float(mean), std, int(total_n)

def get_next_version(config_dir, nickname):
    pattern = os.path.join(config_dir, f"config_{nickname}_v*.json")
    existing = sorted(glob.glob(pattern))

    if not existing:
        return 1

    versions = []
    for fp in existing:
        stem = Path(fp).stem 
        try:
            v = int(stem.split("_v")[-1])
            versions.append(v)
        except ValueError:
            pass

    return max(versions) + 1 if versions else 1

def get_latest_config(config_dir, nickname):
    pattern = os.path.join(config_dir, f"config_{nickname}_v*.json")
    existing = sorted(glob.glob(pattern))

    if not existing:
        return None

    versioned = []
    for fp in existing:
        stem = Path(fp).stem
        try:
            v = int(stem.split("_v")[-1])
            versioned.append((v, fp))
        except ValueError:
            pass

    if not versioned:
        return None

    versioned.sort(key=lambda x: x[0])
    return versioned[-1][1]

In [4]:
run_dir = os.path.join(root_runs_dir, nickname)
dataset_dir = os.path.join(run_dir, f"dataset_{nickname}")
testset_dir = os.path.join(run_dir, f"testset_{nickname}")
config_dir = os.path.join(run_dir, "config_files")

ensure_dir(run_dir)
ensure_dir(dataset_dir)
ensure_dir(testset_dir)
ensure_dir(config_dir)

print("Run directory:", run_dir)
print("Dataset directory:", dataset_dir)
print("Testset directory:", testset_dir)
print("Config directory:", config_dir)

Run directory: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow
Dataset directory: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow
Testset directory: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/testset_yellow
Config directory: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/config_files


In [5]:
mat = loadmat(mat_path)

matrix_lr = mat["dwi_lr"]
matrix_hr = mat["dwi_hr"]
mask_lr = mat["mask_lr"]
mask_hr = mat["mask_hr"]

norm_value = np.array(mat["norm_value"]).T if "norm_value" in mat else None

lr_matrix_bands = matrix_lr.shape[3]
lr_mask_bands = mask_lr.shape[3]

slices_candidates = {
    "matrix_lr": matrix_lr.shape[2],
    "mask_lr": mask_lr.shape[2],
    "matrix_hr": matrix_hr.shape[2],
    "mask_hr": mask_hr.shape[2],
}

hiresl_matrix = [int(matrix_hr.shape[0]), int(matrix_hr.shape[1])]
hiresl_mask = [int(mask_hr.shape[0]), int(mask_hr.shape[1])]

if manual_numband is not None:
    numband = int(manual_numband)
    print(f"[manual] numband = {numband}")
else:
    if lr_matrix_bands != lr_mask_bands:
        print(f"[warning] mismatch en bandas: matrix_lr={lr_matrix_bands}, mask_lr={lr_mask_bands}. Se usará matrix_lr.")
    numband = int(lr_matrix_bands)
    print(f"[mat] numband = {numband}")

if manual_slices is not None:
    slices = int(manual_slices)
    print(f"[manual] slices = {slices}")
else:
    unique_slices = set(slices_candidates.values())
    if len(unique_slices) == 1:
        slices = int(matrix_lr.shape[2])
        print(f"[mat] slices consistentes = {slices}")
    else:
        slices = int(matrix_lr.shape[2])
        print(f"[warning] slices inconsistentes {slices_candidates}. Se usará matrix_lr.shape[2] = {slices}")

if manual_hiresl is not None:
    if not isinstance(manual_hiresl, (list, tuple)) or len(manual_hiresl) != 2:
        raise ValueError(f"manual_hiresl debe ser [H, W], recibido: {manual_hiresl}")
    hiresl_h = int(manual_hiresl[0])
    hiresl_w = int(manual_hiresl[1])
    print(f"[manual] hiresl_h = {hiresl_h}")
    print(f"[manual] hiresl_w = {hiresl_w}")
else:
    if hiresl_matrix != hiresl_mask:
        raise ValueError(f"HR inconsistente: matrix_hr={hiresl_matrix}, mask_hr={hiresl_mask}")
    hiresl_h = int(hiresl_matrix[0])
    hiresl_w = int(hiresl_matrix[1])
    print(f"[mat] hiresl_h = {hiresl_h}")
    print(f"[mat] hiresl_w = {hiresl_w}")

hiresl = [hiresl_h, hiresl_w]
print(f"[final] hiresl = {hiresl}")

[mat] numband = 9
[mat] slices consistentes = 1440
[mat] hiresl_h = 258
[mat] hiresl_w = 258
[final] hiresl = [258, 258]


In [6]:
base_config = load_json(base_config_path)
new_config = deepcopy(base_config)

if "others" not in new_config:
    raise KeyError("El config base no contiene la sección 'others'.")

new_config["others"]["dataset_path"] = dataset_dir + "/"
new_config["others"]["testset_path"] = testset_dir + "/"
new_config["others"]["T_in"] = int(T_in)
new_config["others"]["nickname"] = nickname
new_config["others"]["numband"] = numband
new_config["others"]["hiresl_h"] = hiresl_h
new_config["others"]["hiresl_w"] = hiresl_w
new_config["others"]["slices"] = slices
new_config["others"]["mat_path"] = mat_path
new_config["others"]["num_subj"] = num_subj
new_config["others"]["mat_path"] = mat_path
new_config["others"]["train_subj"] = train_subj
new_config["others"]["registration_ref"] = registration_ref
new_config["others"]["shift_abs_threshold"] = shift_abs_threshold
new_config["others"]["scale"] = scale

if "spectral_band" in new_config["others"]:
    new_config["others"]["spectral_band"] = band_name

In [7]:
latest_config_path = get_latest_config(config_dir, nickname)

if latest_config_path is not None:
    latest_config = load_json(latest_config_path)

    old_hash = canonical_json_hash(latest_config)
    new_hash = canonical_json_hash(new_config)

    if old_hash == new_hash:
        print("\nNo se detectaron cambios respecto a la última versión.")
        print("No se creó un config nuevo.")
        print("Última versión:", latest_config_path)
        config_used_path = latest_config_path
    else:
        next_version = get_next_version(config_dir, nickname)
        config_used_path = os.path.join(config_dir, f"config_{nickname}_v{next_version}.json")
        save_json(config_used_path, new_config)
        print("\nSe detectaron cambios respecto a la última versión.")
        print("Nuevo config guardado en:", config_used_path)
else:
    config_used_path = os.path.join(config_dir, f"config_{nickname}_v1.json")
    save_json(config_used_path, new_config)
    print("\nNo existían versiones previas para este nickname.")
    print("Nuevo config guardado en:", config_used_path)


No existían versiones previas para este nickname.
Nuevo config guardado en: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/config_files/config_yellow_v1.json


In [8]:
print("\n========== RESUMEN ==========")
print("Config base original:", base_config_path)
print("Config usado:", config_used_path)
print("nickname:", new_config["others"]["nickname"])
print("dataset_path:", new_config["others"]["dataset_path"])
print("testset_path:", new_config["others"]["testset_path"])
print("T_in:", new_config["others"]["T_in"])
print("numband:", new_config["others"]["numband"])

print("hiresl_h:", new_config["others"]["hiresl_h"])
print("hiresl_w:", new_config["others"]["hiresl_w"])

print("slices:", new_config["others"]["slices"])
print("mu:", new_config["others"].get("mu"))
print("sigma:", new_config["others"].get("sigma"))
print("sigma_rescaled:", new_config["others"].get("sigma_rescaled"))
print("mat_path:", new_config["others"]["mat_path"])
print("train_subj:", new_config["others"]["train_subj"])
print("registration_ref:", new_config["others"]["registration_ref"])
print("shift_abs_threshold:", new_config["others"]["shift_abs_threshold"])
print("scale:", new_config["others"]["scale"])


========== RESUMEN ==========
Config base original: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/DeepSUM_config_DWIredo.json
Config usado: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/config_files/config_yellow_v1.json
nickname: yellow
dataset_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/dataset_yellow/
testset_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_runs/yellow/testset_yellow/
T_in: 9
numband: 9
hiresl_h: 258
hiresl_w: 258
slices: 1440
mu: 7754.664
sigma: 2316.3408
sigma_rescaled: 596.982
mat_path: /misc/echinocactus/daniellsc/CodeDeepSum/C_DeepSUM/C_global_files/cmats/real_data_DEEPSUM_12subj_9reps_scale3_20251020.mat
train_subj: 10
registration_ref: 1
shift_abs_threshold: 4
scale: 3
